![Universidad Iberoamericana del Ecuador](https://raw.githubusercontent.com/andresrubiop/ml-unibe-estudiantes/main/utils/logo-unibe-color.png)

<div style="background:#004890;color:#FFFFFF;padding:14px 18px;border-radius:10px;margin:6px 0 12px 0;border-bottom:4px solid #FCC000"><div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;opacity:.9">Machine Learning · Sesión 2 · Actividad calificada · aprendizaje autónomo</div><div style="font-size:22px;font-weight:700;margin-top:4px">Deber 2 · Segmentar sin etiquetas: PCA, k-means, mezcla de gaussianas y DBSCAN</div><div style="font-size:12px;opacity:.9;margin-top:4px">Posgrado · Maestría en Big Data y Ciencia de Datos · UNIB.E</div></div>

| | |
|---|---|
| **Qué entregas** | **Este notebook ejecutado** (`.ipynb`), renombrado `D2_Apellido_Nombre.ipynb`, en el aula virtual. No hay que subir nada más. |
| **Fecha** | **Jueves 19 de noviembre de 2026** |
| **Tiempo estimado** | 3 a 4 horas (los modelos tardan unos 5 minutos en total en Colab) |
| **Peso** | 1/3 de los 2 puntos de aprendizaje autónomo · rúbrica al final de este notebook y en el aula |
| **Uso de IA** | **Permitido sin límite**: Claude Code, Codex, ChatGPT, Gemini en Colab, Copilot u otra herramienta, siempre que lo **declares** al final. |
| **Teoría** | Manual M2 §1–§6 · laboratorios L2.1 (k-means y DBSCAN) y L2.2 (dimensionalidad y anomalías) |
| **Datos** | **Fashion-MNIST**: fotos de 28 × 28 de 10 tipos de prendas. La celda de carga ya está hecha. |

**Qué vas a hacer.** Una tienda en línea tiene miles de fotos **sin etiquetar**. Vas a agruparlas como si no supieras qué prenda es cada una, y solo al final usarás las etiquetas para ver qué tan bien lo hiciste.

**La regla de oro de este deber:** las etiquetas `y` **no se usan** hasta la parte 5. Si aparecen antes en tu código, el agrupamiento deja de ser «sin etiquetas».

**Partes:** 0 · cómo trabajar → 1 · configuración → 2 · datos (hecho) → 3 · PCA → 4 · k-means, mezcla de gaussianas y DBSCAN → 5 · comparación con las etiquetas → 6 · conclusión para la tienda → declaración de IA y lista de control.

## 0 · Cómo trabajar este notebook

1. **Guarda tu copia.** En Colab: *Archivo → Guardar una copia en Drive*. Trabaja siempre sobre esa copia.
2. **Ejecuta de arriba hacia abajo.** Las celdas ya resueltas preparan los datos y las funciones de ayuda.
3. **Completa las celdas `# TODO`.** Cada parte explica qué se pide y trae una pista. Las celdas **✅ Autoverificación** usan `assert`: si no muestran error, vas bien.
4. **Responde en las celdas ✍️** con tus palabras.
5. **Antes de entregar:** *Entorno de ejecución → Reiniciar y ejecutar todas*; luego *Archivo → Descargar → Descargar .ipynb*, renómbralo `D2_Apellido_Nombre.ipynb` y súbelo al aula virtual.

## 1 · Configuración

In [ ]:
import gzip
import os
import time
import urllib.request
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from cycler import cycler

from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # avisos de convergencia de la logística: no cambian las conclusiones
SEED = 2026
UNIBE = {"azul": "#004890", "naranja": "#C2410C", "dorado": "#A87900", "verde": "#3A7D44", "gris": "#77787B"}
plt.rcParams.update({
    "axes.prop_cycle": cycler(color=list(UNIBE.values())), "axes.titlecolor": UNIBE["azul"],
    "axes.titleweight": "bold", "axes.edgecolor": UNIBE["gris"], "axes.grid": True, "grid.color": "#E8EDF2",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "legend.frameon": False,
})
CACHE = os.path.expanduser("~/.cache/ml-unibe")
URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/"
print(f"scikit-learn {sklearn.__version__} · semilla {SEED}")

## 2 · Los datos (ya resuelto)

Usamos **10 000 fotos** de Fashion-MNIST elegidas con semilla fija. Las etiquetas quedan guardadas aparte en `y_oculta`: **no las mires** hasta la parte 5.

<details><summary><b>Ayuda: otras formas de cargar Fashion-MNIST</b></summary>

- **Con scikit-learn (OpenML):** `fetch_openml("Fashion-MNIST", version=1, return_X_y=True, as_frame=False)`.
- **Con PyTorch:** `torchvision.datasets.FashionMNIST(root="datos", train=True, download=True)`.
- **Desde tu Google Drive:** sube los cuatro archivos `.gz` a una carpeta de Drive, móntala con `from google.colab import drive; drive.mount("/content/drive")` y cambia `CACHE` por esa carpeta.
</details>

In [ ]:
def descargar(nombre):
    """Descarga un archivo del repositorio público de datasets de Keras (una sola vez)."""
    os.makedirs(CACHE, exist_ok=True)
    ruta = os.path.join(CACHE, nombre)
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(URL + nombre, ruta)
    return ruta


def cargar_fashion():
    """Fashion-MNIST: imágenes y etiquetas de entrenamiento (60 000 fotos de 28 × 28)."""
    with gzip.open(descargar("train-images-idx3-ubyte.gz")) as f:
        X = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)
    with gzip.open(descargar("train-labels-idx1-ubyte.gz")) as f:
        y = np.frombuffer(f.read(), np.uint8, offset=8)
    return X, y


def mostrar(imagenes, titulos=None, n_col=10, tam=1.1):
    """Dibuja una fila (o varias) de imágenes de 28 × 28."""
    n = len(imagenes)
    filas = int(np.ceil(n / n_col))
    fig, axs = plt.subplots(filas, n_col, figsize=(tam * n_col, tam * 1.25 * filas), squeeze=False)
    for k, ax in enumerate(axs.ravel()):
        ax.axis("off")
        if k < n:
            ax.imshow(np.asarray(imagenes[k]).reshape(28, 28), cmap="Greys")
            if titulos is not None:
                ax.set_title(titulos[k], fontsize=7)
    plt.show()


PRENDAS = ["camiseta", "pantalón", "suéter", "vestido", "abrigo", "sandalia", "camisa", "zapatilla", "bolso", "botín"]
X_todo, y_todo = cargar_fashion()
idx = np.random.default_rng(SEED).choice(len(X_todo), 10_000, replace=False)
X = X_todo[idx] / 255.0            # 10 000 fotos × 784 píxeles entre 0 y 1
y_oculta = y_todo[idx]             # NO usar hasta la parte 5
print("datos:", X.shape)
mostrar(X[:20], n_col=10)

## 3 · Reducir la dimensión con PCA

784 píxeles es mucho para agrupar: muchos están siempre en blanco y los vecinos se parecen. PCA encuentra pocas direcciones que conservan casi toda la variación.

**Tu tarea:**
1. Ajusta un `PCA` con todos los componentes y dibuja la **varianza explicada acumulada**.
2. Encuentra `n90`, el **número de componentes que explican el 90 %** de la varianza.
3. **Reconstruye 6 prendas** con 10, 50 y `n90` componentes (`transform` y luego `inverse_transform`) y compáralas con el original.

**Pista:** `np.cumsum(pca.explained_variance_ratio_)` da la curva y `np.argmax(curva >= 0.90) + 1` da `n90`.

In [ ]:
from sklearn.decomposition import PCA

# TODO: ajusta el PCA completo, calcula la curva acumulada y n90.
pca_total = PCA(random_state=SEED).fit(X)
curva = ...
n90 = ...
plt.plot(np.arange(1, len(curva) + 1), curva)
plt.axhline(0.90, ls="--", color=UNIBE["gris"]); plt.axvline(n90, ls="--", color=UNIBE["naranja"])
plt.xlim(0, 300); plt.xlabel("número de componentes"); plt.ylabel("varianza explicada acumulada")
plt.title(f"{n90} componentes explican el 90 %"); plt.show()

In [ ]:
# TODO: reconstruye las 6 primeras fotos con 10, 50 y n90 componentes y muéstralas junto al original.
seis = X[:6]
filas_img, titulos = list(seis), ["original"] * 6
for k in (10, 50, n90):
    pca_k = PCA(n_components=k, random_state=SEED).fit(X)
    reconstruidas = ...
    filas_img += list(reconstruidas)
    titulos += [f"{k} comp."] * 6
mostrar(filas_img, titulos, n_col=6)

**✅ Autoverificación de la parte 3**

In [ ]:
assert 50 <= n90 <= 250, "n90 debería estar entre 50 y 250 componentes para Fashion-MNIST."
assert abs(curva[n90 - 1] - 0.90) < 0.02, "Revisa el cálculo de n90."
print(f"✓ Parte 3 correcta: n90 = {n90}")

Para agrupar usaremos **50 componentes** (`Z50`): conservan la forma de las prendas y hacen todo mucho más rápido.

In [ ]:
Z50 = PCA(n_components=50, random_state=SEED).fit_transform(X)
print("Z50:", Z50.shape)

✍️ **Pregunta 3.** ¿Qué se pierde con 10 componentes y qué se conserva con 50? ¿Por qué tiene sentido agrupar sobre `Z50` y no sobre los 784 píxeles?

*Tu respuesta aquí.*

## 4 · Tres formas de agrupar

### 4.1 · k-means: elegir k con el codo y la silueta

**Tu tarea:** para k de 4 a 15, ajusta `KMeans(n_clusters=k, n_init=5, random_state=SEED)` sobre `Z50` y guarda la **inercia** (para el codo) y la **silueta** (`silhouette_score` con `sample_size=3000` para que sea rápido). Luego elige k y justifícalo.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ks = list(range(4, 16))
inercias, siluetas = [], []
t0 = time.time()
for k in ks:
    # TODO: ajusta k-means con k grupos y guarda su inercia y su silueta.
    km = KMeans(...).fit(Z50)
    inercias.append(...)
    siluetas.append(silhouette_score(Z50, km.labels_, sample_size=3000, random_state=SEED))
print(f"k-means para {len(ks)} valores de k en {time.time() - t0:.0f} s")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.2))
a1.plot(ks, inercias, "o-"); a1.set_title("Codo (inercia)"); a1.set_xlabel("k")
a2.plot(ks, siluetas, "o-", color=UNIBE["naranja"]); a2.set_title("Silueta"); a2.set_xlabel("k")
plt.show()

In [ ]:
# TODO: elige k (puede ser el de mejor silueta o el del codo, pero justifícalo) y muestra los centroides como imágenes.
k_elegido = ...
kmeans = KMeans(n_clusters=k_elegido, n_init=10, random_state=SEED).fit(Z50)
grupos_km = kmeans.labels_
pca50 = PCA(n_components=50, random_state=SEED).fit(X)
centroides = pca50.inverse_transform(kmeans.cluster_centers_)      # de vuelta a 784 píxeles para verlos
mostrar(centroides, [f"grupo {g}" for g in range(k_elegido)], n_col=min(k_elegido, 10))
print("tamaño de cada grupo:", np.bincount(grupos_km))

✍️ **Pregunta 4.1.** ¿Qué k elegiste y por qué? Mirando los centroides, ¿qué prenda «ves» en cada grupo?

*Tu respuesta aquí.*

### 4.2 · Mezcla de gaussianas: elegir K y la covarianza con el BIC

A diferencia de k-means, una mezcla de gaussianas da a cada foto una **probabilidad** de pertenecer a cada grupo.

**Tu tarea:** sobre los primeros **20 componentes** (`Z20`), prueba K de 6 a 14 y los tipos de covarianza `"diag"` y `"full"`. Elige la combinación con **menor BIC**. Luego muestra las **10 fotos más dudosas** (las de menor probabilidad máxima).

**Pista:** `GaussianMixture(n_components=K, covariance_type=tipo, random_state=SEED).fit(Z20).bic(Z20)`.

In [ ]:
from sklearn.mixture import GaussianMixture

Z20 = Z50[:, :20]
resultados_bic = []
t0 = time.time()
for tipo in ("diag", "full"):
    for K in range(6, 15, 2):
        # TODO: ajusta la mezcla y guarda (tipo, K, bic) en resultados_bic.
        gm = GaussianMixture(...).fit(Z20)
        resultados_bic.append({"covarianza": tipo, "K": K, "BIC": ...})
tabla_bic = pd.DataFrame(resultados_bic).sort_values("BIC").reset_index(drop=True)
print(f"{len(tabla_bic)} mezclas en {time.time() - t0:.0f} s")
tabla_bic.head()

In [ ]:
mejor = tabla_bic.loc[0]
gmm = GaussianMixture(n_components=int(mejor["K"]), covariance_type=mejor["covarianza"], random_state=SEED).fit(Z20)
grupos_gmm = gmm.predict(Z20)
# TODO: calcula la probabilidad máxima de cada foto y muestra las 10 más dudosas.
prob_max = gmm.predict_proba(Z20).max(axis=1)
dudosas = ...                       # pista: np.argsort ordena de menor a mayor
mostrar(X[dudosas], [f"p = {prob_max[i]:.2f}" for i in dudosas])

✍️ **Pregunta 4.2.** ¿Qué K y qué covarianza eligió el BIC? ¿Las fotos dudosas tienen algo en común?

*Tu respuesta aquí.*

### 4.3 · DBSCAN: densidad y ruido

DBSCAN no pide k: agrupa las zonas densas y marca como **ruido** (`-1`) lo que queda aislado. En dimensiones altas le cuesta, así que trabaja sobre pocas componentes.

**Tu tarea:** usa los primeros **5 componentes** (`Z5`), dibuja la curva de **k-distancias** (distancia al 10.º vecino, ordenada) y elige `eps` mirando dónde la curva «se dobla». Ajusta `DBSCAN(eps=..., min_samples=10)` y di cuántos grupos encuentra y qué porcentaje marca como ruido.

**Prueba al menos dos valores de `eps`.** Vas a ver el dilema de DBSCAN con estos datos: con un `eps` grande casi todo queda en un solo grupo; con uno pequeño aparecen muchos grupos y mucho ruido. Eso es un resultado, no un error: las prendas no forman zonas densas bien separadas.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

Z5 = Z50[:, :5]
dist, _ = NearestNeighbors(n_neighbors=10).fit(Z5).kneighbors(Z5)
kdist = np.sort(dist[:, -1])
plt.plot(kdist); plt.ylabel("distancia al 10.º vecino"); plt.xlabel("fotos ordenadas"); plt.title("k-distancias"); plt.show()

# TODO: elige eps mirando la curva y ajusta DBSCAN.
eps = ...                            # pista: prueba varios valores, por ejemplo np.quantile(kdist, 0.5) y np.quantile(kdist, 0.9)
grupos_db = DBSCAN(eps=eps, min_samples=10).fit_predict(Z5)
n_grupos_db = len(set(grupos_db)) - (1 if -1 in grupos_db else 0)
print(f"eps = {eps:.2f} · {n_grupos_db} grupos · ruido: {np.mean(grupos_db == -1):.1%}")

**✅ Autoverificación de la parte 4**

In [ ]:
assert len(inercias) == len(ks) and len(siluetas) == len(ks), "Guarda una inercia y una silueta por cada k."
assert len(set(grupos_km)) == k_elegido, "k-means debe tener k_elegido grupos."
assert len(dudosas) == 10 and prob_max[dudosas].max() <= np.sort(prob_max)[9] + 1e-12, "Las dudosas deben ser las 10 de menor probabilidad."
assert n_grupos_db >= 1, "DBSCAN no encontró grupos: prueba un eps mayor."
print("✓ Parte 4 correcta")

## 5 · Ahora sí: comparar con las etiquetas

Recién aquí usamos `y_oculta`. Dos medidas:

- **ARI** (índice de Rand ajustado): 1 = mismo agrupamiento que las etiquetas, 0 = como al azar.
- **Pureza**: en cada grupo, la fracción de fotos de la prenda mayoritaria (la función ya está hecha).

**Tu tarea:** calcula ARI y pureza para k-means, la mezcla de gaussianas y DBSCAN (para DBSCAN, sin contar el ruido), y arma la tabla. Luego mira la **tabla cruzada** de k-means para ver qué prendas se mezclan.

In [ ]:
from sklearn.metrics import adjusted_rand_score


def pureza(grupos, etiquetas):
    """Fracción de fotos que pertenecen a la prenda mayoritaria de su grupo."""
    total = 0
    for g in np.unique(grupos):
        total += np.bincount(etiquetas[grupos == g]).max()
    return total / len(etiquetas)


# TODO: calcula ARI y pureza de los tres agrupamientos.
sin_ruido = grupos_db != -1
comparacion = pd.DataFrame([
    {"método": "k-means", "ARI": adjusted_rand_score(y_oculta, grupos_km), "pureza": pureza(grupos_km, y_oculta)},
    {"método": "mezcla de gaussianas", "ARI": ..., "pureza": ...},
    {"método": "DBSCAN (sin ruido)", "ARI": ..., "pureza": ...},     # usa solo las fotos con grupos_db != -1
])
comparacion.round(3)

In [ ]:
cruzada = pd.crosstab(pd.Series(grupos_km, name="grupo k-means"), pd.Series([PRENDAS[i] for i in y_oculta], name="prenda real"))
cruzada

**✅ Autoverificación de la parte 5**

In [ ]:
assert len(comparacion) == 3, "Compara los tres métodos."
assert comparacion["pureza"].between(0.1, 1).all() and comparacion["ARI"].between(-0.1, 1).all(), "Revisa los cálculos."
assert comparacion["pureza"].max() > 0.4, "Al menos un método debería superar una pureza de 0.4."
print("✓ Parte 5 correcta")

✍️ **Pregunta 5.** ¿Qué `eps` probaste y qué pasó con cada uno? ¿Qué método se acerca más a las prendas reales? ¿Qué prendas se mezclan en un mismo grupo y por qué (forma, tamaño, brillo)? ¿Es «malo» que un grupo junte sandalias y zapatillas?

*Tu respuesta aquí.*

## 6 · Conclusión para la tienda (≈ 150 palabras)

✍️ ¿Para qué le serviría este agrupamiento a una tienda en línea que no tiene sus fotos etiquetadas? Piensa en etiquetar más rápido, detectar fotos raras o recomendar productos parecidos, y menciona una limitación.

*Tu conclusión aquí.*

## Declaración de uso de IA

En este módulo **se permite el uso total de la IA** (Claude Code, Codex, ChatGPT, Gemini, Copilot u otra) **siempre que lo declares**. Completa la tabla aunque no la hayas usado:

| | |
|---|---|
| **Herramienta y versión** | *por ejemplo: Codex, noviembre de 2026* |
| **Para qué la usaste** | *código, depuración, redacción, revisión…* |
| **Prompts principales** | *copia los dos o tres más importantes* |
| **Qué verificaste tú** | *cómo comprobaste que el resultado era correcto* |
| **Qué corregiste** | *errores de la IA que encontraste y cómo los arreglaste* |

## Lista de control antes de entregar

- [ ] *Reiniciar y ejecutar todas* corre sin errores y todas las autoverificaciones muestran ✓.
- [ ] Las etiquetas (`y_oculta`) solo aparecen en la parte 5.
- [ ] Respondiste las preguntas 3, 4.1, 4.2 y 5, la conclusión y la declaración de IA.
- [ ] El archivo se llama `D2_Apellido_Nombre.ipynb` y lo subiste al aula virtual antes del jueves 19/11.

**Rúbrica (10 puntos):** reproducibilidad 1 · PCA aplicado e interpretado 2 · k-means con k justificado 2 · mezcla de gaussianas con BIC e incertidumbre 2 · DBSCAN y comparación final sin usar etiquetas antes de tiempo 2 · conclusión aplicada 1.